# F1 Strategy Predictor - Race Simulator
## Simulazione Pit Stop in Tempo Reale

---

### Obiettivo
Questo notebook simula una gara F1 utilizzando i modelli LSTM addestrati per:

1. **Predire il timing ottimale del pit stop**
   - Output: probabilita di pit per ogni giro
   - Raccomandazione: giro con massima P(pit) per stint

2. **Predire il compound da montare**
   - Output: SOFT, MEDIUM, HARD, INTERMEDIATE, WET
   - Basato sulla sequenza dello stint precedente

### Prerequisiti
- `f1_pit_model.keras`, `f1_compound_model.keras`
- `f1_pit_scaler.pkl`, `f1_comp_scaler.pkl`, `label_encoder.pkl`
- `modelConfig.json`
- `f1_dataset_featured.pkl`

---
# 1. Setup

In [ ]:
# FastF1: libreria open-source per dati F1 (telemetria, tempi, meteo)
import importlib.util
if importlib.util.find_spec('fastf1') is None:
    !pip install fastf1 --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import joblib
from tensorflow.keras.models import load_model

plt.style.use('seaborn-v0_8-whitegrid')

COMPOUND_COLORS = {'SOFT': '#FF3333', 'MEDIUM': '#FFD700', 'HARD': '#E8E8E8',
                   'INTERMEDIATE': '#39FF14', 'WET': '#00BFFF'}
COMPOUND_SHORT = {'SOFT': 'S', 'MEDIUM': 'M', 'HARD': 'H', 'INTERMEDIATE': 'I', 'WET': 'W'}

with open('modelConfig.json', 'r') as f:
    config = json.load(f)

model_pit = load_model('f1_pit_model.keras')
model_comp = load_model('f1_compound_model.keras')
scaler_pit = joblib.load('f1_pit_scaler.pkl')
scaler_comp = joblib.load('f1_comp_scaler.pkl')
label_encoder = joblib.load('label_encoder.pkl')
df_f1 = pd.read_pickle('f1_dataset_featured.pkl')

SEQ_LEN = config['sequence_length']
FEATURES_PIT = config['features_pit']
FEATURES_COMP = config['features_compound']
PIT_THRESHOLD = config['pit_threshold']
WINDOW = 3

print('Modelli caricati')
print(f'Threshold pit: {PIT_THRESHOLD:.3f}')
print(f'Sequenza: {SEQ_LEN} giri')

---
# 2. Selezione Gara

Esegui le celle seguenti per visualizzare le opzioni disponibili, poi configura i parametri.

### 2.1 Anni disponibili

In [ ]:
anni = sorted(df_f1['Year'].unique())
print('Anni disponibili:')
print(', '.join(map(str, anni)))

*Scelta anno:*

In [ ]:
ANNO =

### 2.2 Gare disponibili per anno
Modifica `ANNO` per vedere le gare di quell'anno.

In [ ]:
gare = df_f1[df_f1['Year'] == ANNO][['Round', 'RaceName']].drop_duplicates().sort_values('Round')
print(f'Gare {ANNO}:')
print('-' * 40)
for _, row in gare.iterrows():
    print(f"  {int(row['Round']):>2}. {row['RaceName']}")

*Scelta gara:*

In [ ]:
ROUND =

### 2.3 Piloti disponibili per gara
Modifica `ROUND` per vedere i piloti di quella gara.

In [ ]:
gara_df = df_f1[(df_f1['Year'] == ANNO) & (df_f1['Round'] == ROUND)]
race_name = gara_df['RaceName'].iloc[0] if len(gara_df) > 0 else 'N/A'

piloti = gara_df[['Driver', 'Team']].drop_duplicates().sort_values('Team')
print(f'{race_name} {ANNO}')
print('-' * 40)
print(f'{"Pilota":<6} {"Team":<25}')
print('-' * 40)
for _, row in piloti.iterrows():
    print(f"{row['Driver']:<6} {row['Team']:<25}")

*Scelta pilota:*

In [ ]:
PILOTA = '' # scrivi il nome del pilota nello stile: 'VER'

### 2.4 Configurazione finale

In [ ]:
race_data = df_f1[(df_f1['Year'] == ANNO) & (df_f1['Round'] == ROUND)].copy()
race_name = race_data['RaceName'].iloc[0] if len(race_data) > 0 else None

if race_name and PILOTA in race_data['Driver'].values:
    driver_data = race_data[race_data['Driver'] == PILOTA].sort_values('LapNumber')
    total_race_laps = int(driver_data['LapNumber'].max())
    team = driver_data['Team'].iloc[0]

    stints_info = []
    for stint in sorted(driver_data['Stint'].unique()):
        stint_df = driver_data[driver_data['Stint'] == stint]
        compound = stint_df['Compound'].iloc[0]
        n_laps = int(stint_df['LapNumber'].max()) - int(stint_df['LapNumber'].min()) + 1
        stints_info.append(f"{COMPOUND_SHORT.get(compound, '?')}({n_laps})")

    sc_laps = []
    if 'UnderCaution' in driver_data.columns:
        sc_laps = driver_data[driver_data['UnderCaution'] == 1]['LapNumber'].astype(int).tolist()

    print('=' * 50)
    print(f'  {race_name} {ANNO}')
    print(f'  {PILOTA} - {team}')
    print('=' * 50)
    print(f'  Giri totali: {total_race_laps}')
    print(f'  Stint: {driver_data["Stint"].nunique()}')
    print(f'  Strategia (gomma(giri)): {" > ".join(stints_info)}')
    if sc_laps:
        print(f'  Safety Car: {len(sc_laps)} giri')
    print('=' * 50)
else:
    print('ERRORE: Gara o pilota non trovato')
    print(f'  Anno: {ANNO}, Round: {ROUND}, Pilota: {PILOTA}')

---
# 3. Simulazione

Il simulatore processa ogni giro dello stint, costruisce la sequenza e genera la predizione.

In [ ]:
class RaceSimulator:
    def __init__(self, model_pit, model_comp, scaler_pit, scaler_comp,
                 label_encoder, features_pit, features_comp, seq_len, threshold, window=3):
        self.model_pit = model_pit
        self.model_comp = model_comp
        self.scaler_pit = scaler_pit
        self.scaler_comp = scaler_comp
        self.label_encoder = label_encoder
        self.features_pit = [f for f in features_pit if f in df_f1.columns]
        self.features_comp = [f for f in features_comp if f in df_f1.columns]
        self.seq_len = seq_len
        self.threshold = threshold
        self.window = window

    def _prepare_sequence(self, history_df, features, scaler):
        if len(history_df) == 0:
            return np.zeros((1, self.seq_len, len(features)), dtype=np.float32)
        data = scaler.transform(history_df[features].values.astype(np.float32))
        if len(data) < self.seq_len:
            pad = np.zeros((self.seq_len - len(data), len(features)), dtype=np.float32)
            data = np.vstack([pad, data])
        else:
            data = data[-self.seq_len:]
        return data.reshape(1, self.seq_len, len(features))

    def simulate(self, race_df, driver):
        driver_df = race_df[race_df['Driver'] == driver].sort_values('LapNumber').reset_index(drop=True)
        if len(driver_df) < 2:
            return None, set(), set(), {}, 0

        total_laps = int(driver_df['LapNumber'].max())
        num_stints = int(driver_df['Stint'].max())

        sc_laps = set()
        if 'UnderCaution' in driver_df.columns:
            sc_laps = set(driver_df[driver_df['UnderCaution'] == 1]['LapNumber'].astype(int))

        lap_to_stint, lap_to_compound = {}, {}
        for _, row in driver_df.iterrows():
            lap_to_stint[int(row['LapNumber'])] = int(row['Stint'])
            lap_to_compound[int(row['LapNumber'])] = row['Compound']

        for lap in range(1, total_laps + 1):
            if lap not in lap_to_stint:
                for check in range(lap + 1, total_laps + 1):
                    if check in lap_to_stint:
                        lap_to_stint[lap] = lap_to_stint[check]
                        lap_to_compound[lap] = lap_to_compound[check]
                        break

        stint_data, pit_laps, pit_info = {}, set(), {}
        for stint_num in sorted(driver_df['Stint'].unique()):
            stint_df = driver_df[driver_df['Stint'] == stint_num].sort_values('LapNumber').reset_index(drop=True)
            stint_data[int(stint_num)] = stint_df
            if stint_num > 1:
                prev_df = stint_data.get(int(stint_num - 1))
                if prev_df is not None and len(prev_df) > 0:
                    pit_lap = int(prev_df['LapNumber'].max()) + 1
                    pit_laps.add(pit_lap)
                    pit_info[pit_lap] = {'compound': stint_df['Compound'].iloc[0], 'prev_df': prev_df}

        results = []
        for stint_num in sorted(driver_df['Stint'].unique()):
            stint_num = int(stint_num)
            stint_df = stint_data[stint_num]
            stint_start = int(stint_df['LapNumber'].min())

            if stint_num == 1:
                missing_start = 1
            else:
                missing_start = int(stint_data[stint_num - 1]['LapNumber'].max()) + 1

            for lap in range(missing_start, stint_start):
                is_pit = lap == missing_start and stint_num > 1
                comp_pred, comp_actual = None, None
                if is_pit and lap in pit_info:
                    seq = self._prepare_sequence(pit_info[lap]['prev_df'], self.features_comp, self.scaler_comp)
                    probs = self.model_comp.predict(seq, verbose=0)[0]
                    comp_pred = self.label_encoder.classes_[probs.argmax()]
                    comp_actual = pit_info[lap]['compound']

                results.append({
                    'Lap': lap, 'TotalLaps': total_laps, 'Stint': stint_num,
                    'Compound': lap_to_compound.get(stint_start, 'N/A'),
                    'TyreAge': lap - missing_start + 1, 'PitProb': np.nan, 'PitPred': 0,
                    'PitThisLap': int(is_pit), 'IsFinalLaps': lap > total_laps - self.window,
                    'IsLastStint': stint_num == num_stints,
                    'CompPred': comp_pred, 'CompActual': comp_actual,
                    'UnderCaution': int(lap in sc_laps),
                    'DataAvailable': False, 'MissingReason': 'Pit/Outlap' if is_pit else 'No data'
                })

            for idx in range(len(stint_df)):
                row = stint_df.iloc[idx]
                lap = int(row['LapNumber'])
                history = stint_df.iloc[:idx] if idx > 0 else stint_df.iloc[:0]

                seq = self._prepare_sequence(history, self.features_pit, self.scaler_pit)
                pit_prob = float(self.model_pit.predict(seq, verbose=0)[0, 0])

                comp_pred, comp_actual = None, None
                if lap in pit_info:
                    seq = self._prepare_sequence(stint_df.iloc[:idx+1], self.features_comp, self.scaler_comp)
                    probs = self.model_comp.predict(seq, verbose=0)[0]
                    comp_pred = self.label_encoder.classes_[probs.argmax()]
                    comp_actual = pit_info[lap]['compound']

                results.append({
                    'Lap': lap, 'TotalLaps': total_laps, 'Stint': stint_num,
                    'Compound': row['Compound'], 'TyreAge': int(row['TyreLife']) if pd.notna(row['TyreLife']) else 0,
                    'PitProb': pit_prob, 'PitPred': int(pit_prob >= self.threshold),
                    'PitThisLap': int(lap in pit_laps),
                    'IsFinalLaps': lap > total_laps - self.window,
                    'IsLastStint': stint_num == num_stints,
                    'CompPred': comp_pred, 'CompActual': comp_actual,
                    'UnderCaution': int(row.get('UnderCaution', 0)),
                    'DataAvailable': True, 'MissingReason': None
                })

        result_df = pd.DataFrame(results).sort_values('Lap').reset_index(drop=True)

        recommended = {}
        for s in result_df['Stint'].unique():
            s = int(s)
            if s == num_stints:
                continue
            rows = result_df[(result_df['Stint'] == s) & result_df['DataAvailable'] & ~result_df['IsFinalLaps']]
            if len(rows) > 0:
                idx = rows['PitProb'].idxmax()
                recommended[s] = {'lap': int(rows.loc[idx, 'Lap']), 'prob': rows.loc[idx, 'PitProb']}

        return result_df, pit_laps, sc_laps, recommended, num_stints


simulator = RaceSimulator(model_pit, model_comp, scaler_pit, scaler_comp, label_encoder,
                          FEATURES_PIT, FEATURES_COMP, SEQ_LEN, PIT_THRESHOLD, WINDOW)
sim_df, pit_laps, sc_laps, recommended_pits, num_stints = simulator.simulate(race_data, PILOTA)
print(f'Simulazione completata: {len(sim_df)} giri processati')

---
# 4. Risultati

Confronto tra pit consigliati dal modello e pit reali effettuati.

In [ ]:
print('\n' + '=' * 55)
print('         PIT CONSIGLIATI vs PIT REALI')
print('=' * 55)
max_diff = 0

for stint in range(1, num_stints + 1):
    stint_df = sim_df[sim_df['Stint'] == stint]
    compound = stint_df['Compound'].iloc[0]

    print(f'\nStint {stint}: {compound}')
    print('-' * 55)

    if stint == num_stints:
        print('  Ultimo stint - fine gara')
    elif stint in recommended_pits:
        rec = recommended_pits[stint]
        pit_rows = sim_df[(sim_df['Stint'] == stint + 1) & (sim_df['PitThisLap'] == 1)]

        if len(pit_rows) > 0:
            actual = int(pit_rows['Lap'].iloc[0])
            diff = actual - rec['lap']
            max_diff = max(max_diff, diff)
            comp_actual = pit_rows['CompActual'].iloc[0]
            comp_pred = pit_rows['CompPred'].iloc[0]
            matchP = '✅' if diff <4 else '❌'
            matchC = '✅' if comp_pred == comp_actual else '❌'

            print(f'  Pit Previsto:         Giro {rec["lap"]} (P = {rec["prob"]:.0%})')
            print(f'  Pit Reale:            Giro {actual}     [{matchP}]')
            print(f'\n  Compound Previsto:    Gomma {comp_pred}')
            print(f'  Compound Reale:       Gomma {comp_actual}  [{matchC}]')
    else:
        print('  Dati insufficienti')

print('\n' + '=' * 55)

---
# 5. Grafico

Visualizzazione della probabilita di pit durante la gara.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), height_ratios=[3, 1])
data = sim_df[sim_df['DataAvailable']]
total_laps = int(sim_df['TotalLaps'].iloc[0])
STAR_SIZE = 200
pit_laps_sorted = sorted(list(pit_laps))
ax1 = axes[0]

# Probabilità pit
ax1.plot(data['Lap'], data['PitProb'], 'b-', linewidth=2, label='P(Pit)')
ax1.fill_between(data['Lap'], 0, data['PitProb'], alpha=0.2, color='blue')
ax1.axhline(y=PIT_THRESHOLD, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Soglia ({PIT_THRESHOLD:.0%})')

# Stelle pit reali
for i, pit in enumerate(pit_laps_sorted):
    prob = PIT_THRESHOLD if pit < data['Lap'].min() or pit > data['Lap'].max() else np.interp(pit, data['Lap'].values, data['PitProb'].values)
    ax1.scatter([pit], [prob], color='gold', s=STAR_SIZE, marker='*', zorder=6, edgecolors='black', linewidths=1, label='Pit reale' if i == 0 else '')

# Linee pit predette
for i, info in enumerate(recommended_pits.values()):
    ax1.axvline(x=info['lap'], color='green', linewidth=2, alpha=0.7, label='Pit Predetto' if i == 0 else '')

ax1.set_xlim(1, total_laps)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('P(Pit)')
ax1.set_title(f'{PILOTA} - {race_name} {ANNO}')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
sim_df_sorted = sim_df.sort_values(['Stint', 'Lap'])

# Linee gomme reali
for stint_int, sdata in sim_df_sorted.groupby('Stint'):
    compound = sdata['Compound'].iloc[0]
    color = COMPOUND_COLORS.get(compound, 'gray')
    laps = sdata['Lap'].tolist()
    tyre_age = [lap - laps[0] for lap in laps]
    ax2.plot(laps, tyre_age, color=color, linewidth=4, zorder=2)

# Costruisce il dizionario dei compound predetti per ogni stint
pred_compounds = {}
for pit_lap in sorted(pit_laps):
    pit_row = sim_df[sim_df['Lap'] == pit_lap]
    if len(pit_row) > 0 and pd.notna(pit_row['CompPred'].iloc[0]):
        next_stint = int(pit_row['Stint'].iloc[0])
        pred_compounds[next_stint] = pit_row['CompPred'].iloc[0]

# Linee gomme predette pit-by-pit
pred_pits_sorted = []
for i, info in enumerate(recommended_pits.values()):
    stint_num = i + 2
    if stint_num in pred_compounds:
        pred_pits_sorted.append((info['lap'], pred_compounds[stint_num]))
    else:
        print(f"⚠️ Warning: compound predetto mancante per stint {stint_num}")

pred_pits_sorted = sorted(pred_pits_sorted, key=lambda x: x[0])

for i, (pred_lap, pred_compound) in enumerate(pred_pits_sorted):
    next_lap = pred_pits_sorted[i + 1][0] if i + 1 < len(pred_pits_sorted) else total_laps
    pred_laps = list(range(pred_lap, next_lap + 1))
    pred_tyre_age = list(range(len(pred_laps)))
    color = COMPOUND_COLORS.get(pred_compound, 'gray')
    ax2.plot(pred_laps, pred_tyre_age, color=color, linewidth=2, linestyle='--', zorder=3)

# Stelle pit reali (senza label)
STAR_Y = 1
for pit in pit_laps_sorted:
    ax2.scatter(pit, STAR_Y, color='gold', s=STAR_SIZE, marker='*', zorder=6, edgecolors='black', linewidths=1)

# Linee verticali pit predetti
for info in recommended_pits.values():
    ax2.axvline(x=info['lap'], color='green', linestyle='--', linewidth=1.5, alpha=0.5)

# Legenda semplificata
ax2.plot([], [], color='gray', linewidth=4, label='Reale')
ax2.plot([], [], color='gray', linewidth=2, linestyle='--', label='Predetto')

ax2.set_xlim(1, total_laps)
ax2.set_ylim(0, None)
ax2.set_xlabel('Giro')
ax2.set_ylabel('Età gomma')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# 6. Dettaglio Pit

Analisi giro per giro attorno ai pit stop.

In [ ]:
rec_laps = {info['lap'] for info in recommended_pits.values()}

print("\n" + "="*70)
print("                    DETTAGLIO GIRI PIT STOP")
print("="*70)

for pit_lap in sorted(pit_laps):
    pit_row = sim_df[sim_df['Lap'] == pit_lap]
    if len(pit_row) > 0 and pd.notna(pit_row['CompPred'].iloc[0]):
        comp_pred = pit_row['CompPred'].iloc[0]
        comp_actual = pit_row['CompActual'].iloc[0]
        comp_match = '✅' if comp_pred == comp_actual else '❌'
        header = f"PIT L{pit_lap} | {comp_actual} (pred: {comp_pred} {comp_match})"
    else:
        header = f"PIT L{pit_lap}"

    print(f"\n{header}")
    print("-"*70)
    print(f"{'  Lap':<5}{' S':<3}{'Cmp':<5}{'Age':<5}{'P(Pit)':<8}{'Alert':<7}{'Note':<19}")
    print("-"*70)

    window_df = sim_df[(sim_df['Lap'] >= pit_lap - max_diff) & (sim_df['Lap'] <= pit_lap + 2)].copy()

    for _, row in window_df.iterrows():
        lap = int(row['Lap'])
        is_pit = row['PitThisLap'] == 1

        prob_str = f"{row['PitProb']:.0%}" if row['DataAvailable'] else " - "
        alert = "⚠️" if row['DataAvailable'] and row['PitPred'] == 1 else " "
        sc = "🚨" if row['UnderCaution'] == 1 else " "

        note = ""
        if is_pit:
            note = " ◀ PIT"
        elif lap in rec_laps:
            note = "◀ PIT Consigliato"
        elif not row['DataAvailable']:
            note = f"[{row['MissingReason']}]"

        cmp_short = COMPOUND_SHORT.get(row['Compound'], row['Compound'][:1])
        prefix = ">>" if is_pit else "  "

        print(f"{prefix} {lap:<3}{int(row['Stint']):<3}{cmp_short:<5}{int(row['TyreAge']):<5}{prob_str:<8}{f'{alert}{sc}':<7}{note:<19}")

---
# 7. Salva Risultati

Esporta i risultati in formato CSV e il grafico in PNG.

In [ ]:
dirname = f"{ANNO}_R{ROUND}_{PILOTA}"
os.makedirs(dirname, exist_ok=True)

In [ ]:
# CSV simulazione completa
sim_df.to_csv(os.path.join(dirname, 'simulation.csv'), index=False)

# CSV riepilogo pit
pit_summary = []
for stint in range(1, num_stints + 1):
    row = {'Stint': stint, 'Compound': sim_df[sim_df['Stint'] == stint]['Compound'].iloc[0]}

    if stint == num_stints:
        row.update({'RecLap': None, 'RecProb': None, 'ActualLap': None,
                    'Diff': None, 'NextCompound': None, 'PredCompound': None, 'CompMatch': None})
    elif stint in recommended_pits:
        rec = recommended_pits[stint]
        row['RecLap'] = rec['lap']
        row['RecProb'] = round(rec['prob'], 3)

        pit_rows = sim_df[(sim_df['Stint'] == stint + 1) & (sim_df['PitThisLap'] == 1)]
        if len(pit_rows) > 0:
            actual = int(pit_rows['Lap'].iloc[0])
            row['ActualLap'] = actual
            row['Diff'] = actual - rec['lap']
            row['NextCompound'] = pit_rows['CompActual'].iloc[0]
            row['PredCompound'] = pit_rows['CompPred'].iloc[0]
            row['CompMatch'] = row['NextCompound'] == row['PredCompound']

    pit_summary.append(row)

pd.DataFrame(pit_summary).to_csv(os.path.join(dirname, 'summary.csv'), index=False)

# PNG grafico
fig.savefig(os.path.join(dirname, 'chart.png'), dpi=150, bbox_inches='tight')
print('File salvati:')
print(f'  - {dirname}/simulation.csv')
print(f'  - {dirname}/summary.csv')
print(f'  - {dirname}/chart.png')